# Suite MUT — Mutations: the schemas of the verbs a transaction applies

`Mutations` holds the schemas of the mutations a transaction applies ([Serialization](../../docs/SERIALIZATION.md),
Transactions on the wire): an object's verbs (`OfObject`, its properties' and its adjacencies'), an adjacency's
(`OfAdjacency`), value verbs nested in them, the nouns `Ref`, `Def` and `Slice`, and a pattern's spec, `OfPattern`.
Mutations are schema-governed: every store holds their schemas, and mutations are built and written as any value is.
Every store also holds the basic scalars, `Schemas.OfBool` to `Schemas.OfBytes`.

In [ ]:
import json
from types import SimpleNamespace

from mbse.Schemas.Framework import Builders as B, JSON, Plain
from support import raises

store = B.Store()
M = SimpleNamespace(  # the store's own mutation schemas, through its paths
    OfPattern=store.Mutations.OfPattern.as_schema(), OfObject=store.Mutations.OfObject.as_schema(),
    OfAdjacency=store.Mutations.OfAdjacency.as_schema(),
    OfAny=store.Mutations.OfAny.as_schema(), OfContainer=store.Mutations.OfContainer.as_schema(),
    OfLink=store.Mutations.OfLink.as_schema(), PropertySet=store.Mutations.OfProperty.Set.as_schema(),
    AnySet=store.Mutations.OfAny.Set.as_schema(), ScalarSet=store.Mutations.OfScalar.Set.as_schema(),
    SCHEMAS=[store.registered(name) for name in store.names() if name.startswith("Mutations.")])


def written_at(singleton, *names):
    """A path as plain data: from `singleton`, then, for a schema object's, through each module and then the schema by
    name (the root module is the singleton ".")."""
    nested = {"target": {}}
    for name in reversed(names):
        nested = {"entry": {"i": "schemas", "o": "", "properties": {"name": name}, "next": nested}}
    return {"singleton": singleton, "next": nested}


def plain(value, schema=M.OfPattern):
    return Plain.ToPlain(store)(schema, value)

## MUT-01 · The mutation schemas, valid, and the basic scalars, held by every store

In [ ]:
assert all(schema.validate() == [] for schema in M.SCHEMAS) and len(M.SCHEMAS) == 65  # a spec's, mutation expressions' (TXN), a transaction's session and where (PRV)
for schema in M.SCHEMAS:
    assert store.registered(schema.name) is schema
other = B.Store()
assert other.registered("Mutations.OfPattern") is not M.OfPattern and other.Mutations.OfPattern.as_schema().validate() == []  # each its own
for name in ("OfBool", "OfInt", "OfFloat", "OfStr", "OfBytes"):
    scalar = getattr(store.Schemas, name).as_schema()
    assert scalar.name == f"Schemas.{name}" and store.registered(scalar.name) is scalar
assert store.Schemas.OfStr.as_schema().type is str and store.Schemas.OfObject.as_schema().properties["ref"].schema is store.Schemas.OfBool.as_schema()
assert [b.name for b in M.OfObject.branches] == ["as_property", "as_adjacency", "spec"]  # a pattern's own spec too
assert [b.name for b in M.OfAdjacency.branches] == ["insert", "remove", "clear"]
assert [b.name for b in M.OfAny.branches] == ["set", "clear", "as_scalar", "as_container", "as_object", "as_union",
                                             "as_intersection"]
assert M.PropertySet is not M.AnySet is not M.ScalarSet  # each family its own

## MUT-02 · An object's verbs: a property set, and changed through the nested value verbs; an object defined where it is linked

In [ ]:
aged = store.Mutations.OfObject(lambda u: u.as_property(lambda p: (
    p
    .name("age")
    .mutations([store.Mutations.OfProperty(lambda v: v.set(lambda s: s.value(37)))])
)))
deep = store.Mutations.OfObject(lambda u: u.as_property(lambda p: (
    p
    .name("address")
    .mutations([store.Mutations.OfProperty(lambda v: v.as_any(lambda a: a.mutations([
        store.Mutations.OfAny(lambda x: x.as_object(lambda o: o.mutations([
            store.Mutations.OfObject.AsProperty(lambda q: q.name("city").mutations([
                store.Mutations.OfProperty(lambda w: w.set(lambda s: s.value("Paris"))),
            ])),
        ]))),
    ])))])
)))
defined = store.Mutations.OfLink(lambda k: k.set(lambda s: getattr(s, "def")(lambda d: (  # Python cannot spell `.def`
    d
    .schema("crm.Address")
    .mutations([store.Mutations.OfObject(lambda u: u.as_property(lambda p: p.name("city").mutations([
        store.Mutations.OfProperty(lambda v: v.set(lambda s: s.value("Paris")))])))])
))))
written = plain(store.Mutations.OfPattern(lambda c: [c.at("ada").as_container(lambda l: [
    l.at(0).as_union(aged), l.at(1).as_union(deep)])]))
assert written["ada"][0] == {"as_property": {"name": "age", "mutations": [{"set": {"value": {"int": 37}}}]}}
nested = written["ada"][1]["as_property"]["mutations"][0]["as_any"]["mutations"][0]["as_object"]["mutations"][0]
assert nested == {"name": "city", "mutations": [{"set": {"value": {"str": "Paris"}}}]}
assert plain(defined, M.OfLink) == {"set": {"def": {"schema": "crm.Address", "mutations": [
    {"as_property": {"name": "city", "mutations": [{"set": {"value": {"str": "Paris"}}}]}}]}}}

## MUT-03 · Value verbs: containers by slices, by count, and item by item in modulo order; scalars; unions and intersections

In [ ]:
items = store.Mutations.OfContainer(lambda c: c.as_items(lambda i: i.mutations([
    store.Mutations.OfAny(lambda a: a.as_scalar(lambda s: s.mutations([
        store.Mutations.OfScalar(lambda x: x.set(lambda v: v.value(0))),
    ]))),
    store.Mutations.OfAny(lambda a: a.clear(store.Mutations.OfAny.Clear(lambda c: c))),
])))
edits = [
    store.Mutations.OfContainer(lambda c: c.insert(lambda i: i.slice(lambda p: p.as_object(lambda s: s.index(0).items(["a", "b"]))))),
    store.Mutations.OfContainer(lambda c: c.remove(lambda r: r.index(1).count(2))),
    store.Mutations.OfContainer(lambda c: c.append(lambda a: a.items(["z"]))),
    store.Mutations.OfContainer(lambda c: c.prepend(lambda a: a.items(["y"]))),
    store.Mutations.OfContainer(lambda c: c.clear(store.Mutations.OfContainer.Clear(lambda c: c))),
    items,
]
held = [plain(edit, M.OfContainer) for edit in edits]
assert [list(h) for h in held] == [["insert"], ["remove"], ["append"], ["prepend"], ["clear"], ["as_items"]]
assert held[1] == {"remove": {"index": {"int": 1}, "count": 2}} and held[4] == {"clear": {}}
assert held[5]["as_items"]["mutations"][1] == {"clear": {}}
branch = store.Mutations.OfAny(lambda a: a.as_union(lambda u: u.mutations([
    store.Mutations.OfUnion.AsBranch(lambda b: b.name("circle").mutations([
        store.Mutations.OfProperty(lambda p: p.clear(store.Mutations.OfProperty.Clear(lambda c: c))),
    ])),
])))
part = store.Mutations.OfAny(lambda a: a.as_intersection(lambda i: i.mutations([
    store.Mutations.OfIntersection.AsPart(lambda b: b.name("stamp").mutations([])),
])))
assert plain(branch, M.OfAny) == {"as_union": {"mutations": [{"name": "circle", "mutations": [{"clear": {}}]}]}}
assert plain(part, M.OfAny) == {"as_intersection": {"mutations": [{"name": "stamp", "mutations": []}]}}

## MUT-04 · Adjacencies: entries inserted and removed with their links and properties; a link referring by path or by symbol, or cleared

In [ ]:
bob = store.Mutations.OfObject.Ref(lambda r: r.symbol("bob"))
knows = store.Mutations.OfObject(lambda u: u.as_adjacency(lambda a: a.name("acquaintances").mutations([
    store.Mutations.OfAdjacency(lambda j: j.insert(lambda i: i.entries([
        store.Mutations.OfEntry(lambda e: e.mutations([
            store.Mutations.OfEntry.Member(lambda m: m.as_link(lambda l: l.name("known").mutations([
                store.Mutations.OfLink(lambda k: k.set(lambda s: s.ref(lambda f: f.path(lambda p: (  # by its path
                    p.singleton("crm.Bob").next(lambda n: n.target(lambda t: t))))))),
            ]))),
            store.Mutations.OfEntry.Member(lambda m: m.as_property(lambda p: p.name("since").mutations([
                store.Mutations.OfProperty(lambda v: v.set(lambda s: s.value(2020))),
            ]))),
        ])),
        store.Mutations.OfEntry(lambda e: e.mutations([
            store.Mutations.OfEntry.Member(lambda m: m.as_link(lambda l: l.name("known").mutations([
                store.Mutations.OfLink(lambda k: k.set(lambda s: s.ref(bob))),  # by its symbol
            ]))),
            store.Mutations.OfEntry.Member(lambda m: m.as_property(lambda p: p.name("since").mutations([
                store.Mutations.OfProperty(lambda v: v.set(lambda s: s.value(2021))),
            ]))),
        ])),
    ]))),
    store.Mutations.OfAdjacency(lambda j: j.remove(lambda r: r.entries([
        store.Mutations.OfEntry(lambda e: e.mutations([
            store.Mutations.OfEntry.Member(lambda m: m.as_link(lambda l: l.name("known").mutations([
                store.Mutations.OfLink(lambda k: k.set(lambda s: s.ref(bob))),
            ]))),
            store.Mutations.OfEntry.Member(lambda m: m.as_property(lambda p: p.name("since").mutations([
                store.Mutations.OfProperty(lambda v: v.set(lambda s: s.value(2021))),
            ]))),
        ])),
    ]))),
    store.Mutations.OfAdjacency(lambda j: j.clear(store.Mutations.OfAdjacency.Clear(lambda c: c))),
])))
unlinked = store.Mutations.OfLink(lambda k: k.clear(store.Mutations.OfLink.Clear(lambda c: c)))
written = plain(store.Mutations.OfPattern(lambda c: [c.at("ada").as_container(lambda l: [
    l.at(0).as_union(knows)])]))["ada"][0]["as_adjacency"]
assert written["name"] == "acquaintances" and [list(w) for w in written["mutations"]] == [["insert"], ["remove"], ["clear"]]
assert written["mutations"][2] == {"clear": {}}  # every entry it holds
first, second = written["mutations"][0]["insert"]["entries"]
assert first["mutations"][0] == {"as_link": {"name": "known", "mutations": [{"set": {"ref": {"path": written_at("crm.Bob")}}}]}}
assert first["mutations"][1] == {"as_property": {"name": "since", "mutations": [{"set": {"value": {"int": 2020}}}]}}
assert second["mutations"][0]["as_link"]["mutations"] == [{"set": {"ref": {"symbol": "bob"}}}]
assert plain(unlinked, M.OfLink) == {"clear": {}}

## MUT-05 · Mutations read back from plain data and from JSON, as any value is

In [ ]:
named = store.Mutations.OfObject(lambda u: u.as_property(lambda p: p.name("name").mutations([
    store.Mutations.OfProperty(lambda v: v.set(lambda s: s.value("Ada")))])))
written = plain(store.Mutations.OfPattern(lambda c: [c.at("ada").as_container(lambda l: [l.at(0).as_union(named)])]))
back = Plain.FromPlain(store)(M.OfPattern, written)
assert plain(back) == written
text = JSON.ToJSON(store)(M.OfPattern, back)
assert json.loads(text) == written and plain(JSON.FromJSON(store)(M.OfPattern, text)) == written
with raises(Exception):
    Plain.FromPlain(store)(M.OfObject, {"as_property": {}, "as_adjacency": {}})  # a union value holds one branch

## MUT-06 · A spec's verbs: an object's properties and adjacencies, an adjacency's entries inserted, each a `Mutations.OfPattern` by symbol; its payloads bare

In [ ]:
def crm(txn):
    """crm.Person; crm.Book, a singleton holding its contacts."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Holds": S.OfRelation, "Person": S.OfObject, "Book": S.OfObject})
        .spec([
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")]),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr}),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
        ])]))])


def filing(txn):
    """crm.Filing, a pattern over what `crm` installed: Ada, one of the book's contacts."""
    return txn.uid("filing/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Filing": txn.Schemas.OfPattern})
        .spec([lambda _: _.Filing.name("crm.Filing").symbols({"book": txn.crm.Book, "ada": txn.crm.Person}).spec([
            lambda _: _.book.contacts([_.ada]), lambda _: _.ada.name("Ada")])])]))])


spec6 = B.Store()
spec6.transaction(crm)
spec6.transaction(filing)
pattern6 = spec6.registered("crm.Filing")
M6 = spec6.Mutations
contact6 = M6.OfObject(lambda m: m.as_adjacency(lambda a: a.name("contacts").mutations([
    M6.OfAdjacency(lambda j: j.insert(lambda i: i.entries([M6.OfEntry(lambda e: e.mutations([
        M6.OfEntry.Member(lambda x: x.as_link(lambda l: l.name("contact").mutations([
            M6.OfLink(lambda k: k.set(lambda s: s.ref(M6.OfObject.Ref(lambda r: r.symbol("ada"))))),
        ]))),
    ]))])))])))
named6 = M6.OfObject(lambda m: m.as_property(lambda p: p.name("name").mutations([
    M6.OfProperty(lambda v: v.set(lambda s: s.value("Ada")))])))
built6 = M6.OfPattern(lambda c: [c.at("book").as_container(lambda l: [l.at(0).as_union(contact6)]),
                                 c.at("ada").as_container(lambda l: [l.at(0).as_union(named6)])])
written6 = Plain.ToPlain(spec6)(M6.OfPattern, built6)
assert Plain.ToPlain(spec6)(M6.OfPattern, Plain.FromPlain(spec6)(M6.OfPattern, written6)) == written6  # as any value
assert written6["ada"][0]["as_property"]["mutations"][0]["set"]["value"] == {"str": "Ada"}  # tagged, as any value is
written6["ada"][0]["as_property"]["mutations"][0]["set"]["value"] = "Ada"  # a spec's payload is bare, as it records it
assert [dict(item) for item in pattern6.spec] == [{"book": written6["book"]}, {"ada": written6["ada"]}]  # what it records
with raises(Exception):
    Plain.FromPlain(spec6)(M6.OfPattern, {"ada": [{"as_property": {}, "as_adjacency": {}}]})  # one verb each